<a href="https://colab.research.google.com/github/parthibanxd/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parthibanxd/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh / Content Opportunity Scoring, framed as a ranking problem. The goal is to score existing content items by how strongly they appear to deserve review, so an SEO or editorial team can inspect the highest-priority pages first. Ranking is more appropriate than simple classification because the practical decision is not only whether a page is declining, but which pages should receive limited review time first.

In [ ]:
task_type = "Ranking / Scoring"

print("ML task type:", task_type)
print("Decision: Which content pages should be reviewed first?")

ML task type: Ranking / Scoring
Decision: Which content pages should be reviewed first?


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter dataset provides trend_direction, which can be converted into is_declining_proxy where trend_direction == "down". I will use this as a starter proxy for content deterioration, not as a true future prediction target. It is derived from the current observation window, so it does not prove that a page will decline later. A stronger future target would use historical features to predict an observed decline or recovery during a subsequent future window.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/parthibanxd/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df["is_declining_proxy"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Declining proxy:", df["is_declining_proxy"].sum())
print("Declining proxy rate:", f"{df['is_declining_proxy'].mean():.1%}")

Rows: 30000
Declining proxy: 16262
Declining proxy rate: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric is Precision@50. The intended action is to review the highest-ranked pages first, so the important question is how many of the top 50 ranked pages are actually associated with the observed declining proxy. Precision@50 directly measures whether useful candidates are placed near the top of the review queue. This is more appropriate than overall accuracy because the real-world action is prioritization rather than assigning every page a yes/no decision.

In [ ]:
k = 50

print("Primary metric: Precision@50")
print("Top-K pages reviewed:", k)

Primary metric: Precision@50
Top-K pages reviewed: 50


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one content item/page. Each row represents one existing content item and contains observable signals such as search impressions, clicks, sessions, average position, CTR, content age, update age, word count, engagement, and trend information. The ranking score operates at the page level because the downstream action is to decide which individual pages an SEO specialist or content editor should review first.

In [ ]:
lane_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
    "is_declining_proxy",
]

lane_df = df[lane_cols].copy()

print("One row = one content item/page")
print("Rows:", len(lane_df))
print("Columns:", len(lane_df.columns))

lane_df.head(10)

One row = one content item/page
Rows: 30000
Columns: 14


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,3803,29,17,10.6,0.76,5.88,4.55,187,20,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,20.3,0.05,0.00,10.00,445,25,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,36.5,0.09,0.00,28.57,141,20,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,78,6.2,0.49,1.28,3.45,463,22,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,44.0,0.13,0.00,24.29,263,14,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,5,8.5,0.03,0.00,25.00,147,20,3080.0,down,1
6,content_9a34b442b552,client_8722616204,20,0,1,7.0,0.00,0.00,0.00,90,20,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,1724,1,28,21.2,0.06,3.57,7.14,445,22,NaN,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,68,46.0,0.09,5.88,6.25,90,20,3807.0,down,1
9,content_c27558df2b0c,client_19581e27de,1240,2,3,4.9,0.16,0.00,0.00,257,104,NaN,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can identify obvious cases, such as pages that are old and declining. However, refresh opportunity depends on several interacting signals including search visibility, traffic, ranking position, CTR, content age, update age, content depth, engagement, and trend. These relationships can be difficult to capture with a few manually chosen thresholds. ML can learn patterns across multiple signals and produce a ranked review queue. A fixed rule should still be used as a baseline, and ML should only be preferred if it consistently improves Precision@50 under honest validation.

In [ ]:
candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

print("Candidate signals:", len(candidate_features))
print("\nSignals:")
for feature in candidate_features:
    print("-", feature)

Candidate signals: 10

Signals:
- impressions_90d
- clicks_90d
- sessions_90d
- avg_position
- ctr
- engagement_rate
- scroll_rate
- content_age_days
- days_since_last_update
- word_count


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.